# Debugging Non-Deterministic Systems

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/10-debugging-non-deterministic-systems.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A traditional software bug is reproducible. You call a function with input X, it returns wrong output Y, you trace the code, you find the bug, you fix it. Run the test again: it passes. Done.

AI systems break differently. Your summarizer returns a wrong answer on Tuesday. You run the same query on Wednesday and it returns the right answer. You check the code: nothing changed. The model is probabilistic. The Tuesday failure might have been a one-in-ten event. It might have been a one-in-thousand event. Without a log, you cannot tell.

This is the central challenge of AI debugging: you are debugging a distribution, not a function. The question is never just "did it fail?" It is "at what rate ...

## The Concept

### Deterministic vs. Probabilistic Debugging

```
DETERMINISTIC (traditional software)       PROBABILISTIC (AI systems)
-----------------------------------------  ----------------------------------------
Same input → same output always            Same input → different output each run
Bug: wrong output for specific input       Bug: wrong output at unacceptable rate
Reproduce: re-run with same args           Reproduce: need temperature=0 + same prompt
Debug: read the stack trace                Debug: classify failures across many runs
Fix: change the code                       Fix: change prom...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 10 - Debugging Non-Deterministic Systems
A DebugLogger that captures every AI call to JSONL for analysis.

Run:
    python main.py

This makes 3 calls and writes ai_calls.jsonl, then prints a summary.
"""

import anthropic
import json
import os
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional


# ─────────────────────────────────────────────
# Cost estimates per 1K tokens (input / output)
# ─────────────────────────────────────────────

COST_PER_1K: dict[str, dict[str, float]] = {
    "claude-3-5-haiku-20241022": {"input": 0.001, "output": 0.005},
    "claude-3-5-sonnet-20241022": {"input": 0.003, "output": 0.015},
    "claude-opus-4-5": {"input": 0.015, "output": 0.075},
}


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    rates = COST_PER_1K.get(model, {"input": 0.003, "output": 0.015})
    return (input_tokens / 1000) * rates["input"] + (output_tokens / 1000) * rates["output"]


# ─────────────────────────────────────────────
# Data record
# ─────────────────────────────────────────────

@dataclass
class CallRecord:
    timestamp: str
    model: str
    prompt: str
    response: str
    latency_ms: int
    input_tokens: int
    output_tokens: int
    cost_usd: float
    temperature: float
    error: Optional[str] = None

    def to_jsonl(self) -> str:
        return json.dumps(asdict(self))


# ─────────────────────────────────────────────
# Logger
# ─────────────────────────────────────────────

class DebugLogger:
    """
    Wraps the Anthropic client and logs every call to a JSONL file.

    Each line in the file is a JSON record with:
      timestamp, model, prompt, response, latency_ms,
      input_tokens, output_tokens, cost_usd, temperature, error
    """

    def __init__(self, log_path: str = "ai_calls.jsonl"):
        self.log_path = Path(log_path)
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            print("Error: ANTHROPIC_API_KEY not set", file=sys.stderr)
            sys.exit(1)
        self.client = anthropic.Anthropic(api_key=api_key)

    def call(
        self,
        prompt: str,
        model: str = "claude-3-5-haiku-20241022",
        max_tokens: int = 256,
        temperature: float = 1.0,
    ) -> str:
        """Make an API call, log everything, and return the response text."""
        start = time.monotonic()
        error: Optional[str] = None
        response_text = ""
        input_tokens = 0
        output_tokens = 0

        try:
            response = self.client.messages.create(
                model=model,
                max_tokens=max_tokens,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}],
            )
            response_text = response.content[0].text
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens
        except Exception as e:
            error = str(e)

        latency_ms = int((time.monotonic() - start) * 1000)

        record = CallRecord(
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            model=model,
            prompt=prompt,
            response=response_text,
            latency_ms=latency_ms,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            cost_usd=estimate_cost(model, input_tokens, output_tokens),
            temperature=temperature,
            error=error,
        )

        with open(self.log_path, "a") as f:
            f.write(record.to_jsonl() + "\n")

        if error:
            raise RuntimeError(f"API call failed: {error}")

        return response_text


# ─────────────────────────────────────────────
# Log analysis
# ─────────────────────────────────────────────

def load_log(log_path: str = "ai_calls.jsonl") -> list[dict]:
    path = Path(log_path)
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def failure_rate(records: list[dict]) -> float:
    if not records:
        return 0.0
    failures = sum(1 for r in records if r.get("error") is not None)
    return failures / len(records)


def slow_calls(records: list[dict], threshold_ms: int = 3000) -> list[dict]:
    return [r for r in records if r["latency_ms"] > threshold_ms]


def total_cost(records: list[dict]) -> float:
    return sum(r["cost_usd"] for r in records)


def classify_failures(records: list[dict]) -> dict[str, list[dict]]:
    """
    Separate failures into model failures vs. integration failures.
    Integration: rate limits, network errors, auth problems.
    Model: refusals, short/empty responses, wrong format.
    """
    categories: dict[str, list[dict]] = {
        "integration_rate_limit": [],
        "integration_auth": [],
        "integration_network": [],
        "model_refusal": [],
        "model_short_response": [],
        "unknown": [],
    }

    for r in records:
        error = (r.get("error") or "").lower()
        response = r.get("response", "")

        if "rate_limit" in error or "rate limit" in error:
            categories["integration_rate_limit"].append(r)
        elif "authentication" in error or "api_key" in error or "401" in error:
            categories["integration_auth"].append(r)
        elif "connection" in error or "timeout" in error or "network" in error:
            categories["integration_network"].append(r)
        elif response and any(p in response.lower() for p in ["i cannot", "i can't", "i'm unable", "i am unable"]):
            categories["model_refusal"].append(r)
        elif r.get("error") is None and len(response) < 10:
            categories["model_short_response"].append(r)
        elif r.get("error") is not None:
            categories["unknown"].append(r)

    return {k: v for k, v in categories.items() if v}


def print_summary(log_path: str = "ai_calls.jsonl") -> None:
    records = load_log(log_path)
    if not records:
        print("No records found.")
        return

    latencies = [r["latency_ms"] for r in records]
    print(f"Total calls:     {len(records)}")
    print(f"Failure rate:    {failure_rate(records):.1%}")
    print(f"Avg latency:     {sum(latencies) / len(latencies):.0f} ms")
    print(f"Max latency:     {max(latencies)} ms")
    print(f"Slow calls:      {len(slow_calls(records))} (>3000ms)")
    print(f"Total cost:      ${total_cost(records):.4f}")

    categories = classify_failures(records)
    if categories:
        print(f"\nFailure breakdown:")
        for category, items in categories.items():
            print(f"  {category}: {len(items)}")
    else:
        print("\nNo failures recorded.")


# ─────────────────────────────────────────────
# Demo
# ─────────────────────────────────────────────

DEMO_PROMPTS = [
    "Summarize in one sentence: The transformer uses self-attention instead of recurrence.",
    "List three benefits of containerization for AI apps. Be concise.",
    "What is the primary difference between a Python script and a FastAPI service?",
]


def main() -> None:
    log_path = "ai_calls.jsonl"
    logger = DebugLogger(log_path=log_path)

    print("Making 3 calls with DebugLogger...\n")

    for i, prompt in enumerate(DEMO_PROMPTS, 1):
        print(f"Call {i}: {prompt[:70]}")
        try:
            response = logger.call(prompt, temperature=1.0)
            print(f"  -> {response[:100].strip()}")
        except RuntimeError as e:
            print(f"  -> ERROR: {e}")
        print()

    print("=" * 55)
    print_summary(log_path)
    print()
    print(f"Full log: {log_path}")
    print("Each line is a complete JSON record.")
    print()
    print("To reproduce a specific call at temperature=0:")
    print('  logger.call(records[0]["prompt"], temperature=0.0)')

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the correct first step?**

- A. Change the prompt based on the user's description of the wrong answer.
- B. Add a DebugLogger that captures every call before doing any other investigation.
- C. Upgrade to a larger model; wrong answers are always caused by underpowered models.
- D. Roll back the code to Monday's version since it was working then.

**2. Did the prompt change fix the problem?**

- A. Yes: refusals dropped from 4 to 1, which is a clear improvement.
- B. Unclear: you need to compare failure rates: 4/200 = 2% vs 1/200 = 0.5%. The improvement is real but you need to set an acceptable threshold to know if it is enough.
- C. No: you need zero refusals to confirm a fix.
- D. Yes: any reduction in failures confirms the fix.

**3. What type of failure is this, and is your teammate's diagnosis correct?**

- A. Model failure. The teammate is correct; the prompt needs to be rewritten.
- B. Integration failure. HTTP 429 is a rate limit error from the API provider, not a model quality issue. Fix it with retry logic and backoff.
- C. Unknown failure. 429 could mean either a prompt problem or a rate limit problem.
- D. Model failure. 429 means the model rejected the prompt content.

**4. What does this tell you?**

- A. The original failure was a bug in your logging code.
- B. The failure is probabilistic: it occurred at some probability, not systematically. You need to measure the failure rate across many runs.
- C. The failure was caused by a high temperature, and setting temperature=0 permanently will fix it.
- D. The model was updated between the original call and your reproduction attempt.

**5. What is the minimum you need to determine whether this is a real systematic problem?**

- A. 3 examples are sufficient; they prove the bug exists.
- B. The failure rate across all financial documents over the two-week period. Without logs, you cannot determine this.
- C. A manual review of 10 random financial documents from today.
- D. Re-running the 3 failing examples with temperature=0 to confirm they still fail.

**6. Which failures should be addressed first, and with what type of fix?**

- A. The short responses (30 calls) first; model quality issues are harder to fix than timeouts.
- B. Both are equal priority. Fix them at the same time.
- C. The timeouts (50 calls) first via retry/backoff logic (integration fix), then investigate short responses by examining the prompts that triggered them (model fix).
- D. Neither needs fixing; 80 failures out of 930 total (8.6%) is within acceptable range.

**7. What does the JSONL prompt log provide that the Datadog dashboard does NOT?**

- A. Nothing; dashboards and logs are equivalent observability tools.
- B. The exact prompt and response for the specific call that failed, enabling root-cause reproduction and analysis.
- C. Cost tracking; Datadog cannot track per-call API costs.
- D. Latency data; Datadog only tracks HTTP response times, not AI model latency.

**8. **

- A. Ask several users if they noticed an improvement.
- B. Count total errors before and after: if before > after, the change helped.
- C. Compare failure rates: failures/total for each period. Also compare rates by failure type to confirm the specific type you targeted decreased.
- D. Run the 5 worst-performing prompts from before and check if they are better after.

_Answers are in `checks.json` in the lesson directory._